# Phase 1: Pattern Extraction

Extract all unique traits and actions from monster data, identify patterns and frequencies.

**Outputs:**
- `../data/trait_catalog.parquet`
- `../data/action_catalog.parquet`
- `../data/pattern_frequencies.csv`

In [69]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter, defaultdict
from pathlib import Path



In [70]:
# Setup paths
MAIN_MAIN_DATA_DIR = Path('../data')

# Get the current working directory
cwd = Path.cwd()

if cwd.name == 'notebooks/exploration/feature_classification/notebooks':
    MAIN_DATA_DIR = '../../../../data'
    LOCAL_DATA_DIR = '../data'
else:
    MAIN_DATA_DIR = './data'
    LOCAL_DATA_DIR = './notebooks/exploration/feature_classification/data'
# Load monster data
monsters_df = pd.read_csv(MAIN_DATA_DIR + '/dnd5e_monsters_from_json.csv')
# Filter to only 2014 (original 5e) monsters - exclude 2024 additions
monsters_df = monsters_df[monsters_df['Source_Name'] == 'Free Basic Rules (2014)'].copy()
print(f"📊 Loaded {len(monsters_df)} monsters (2014 5e only, excluding 2024 additions)")
print(f"Loaded {len(monsters_df)} monsters")

📊 Loaded 324 monsters (2014 5e only, excluding 2024 additions)
Loaded 324 monsters


## Section 1: Extract All Traits

In [71]:
def extract_traits(df):
    """Extract all traits from monsters into a flat dataframe."""
    all_traits = []
    
    for idx, row in df.iterrows():
        monster_name = row['Name']
        cr = row.get('CR', 'Unknown')
        
        traits_str = row.get('Traits', '')
        if pd.isna(traits_str) or traits_str == '' or traits_str == '—':
            continue
            
        try:
            traits = json.loads(traits_str)
            for trait in traits:
                if isinstance(trait, dict):
                    all_traits.append({
                        'monster': monster_name,
                        'cr': cr,
                        'name': trait.get('Name', 'Unknown'),
                        'description': trait.get('Desc', trait.get('Description', '')),
                        'source': 'Traits'
                    })
        except (json.JSONDecodeError, TypeError):
            continue
    
    return pd.DataFrame(all_traits)

traits_df = extract_traits(monsters_df)
print(f"Extracted {len(traits_df)} total trait instances")
print(f"Unique trait names: {traits_df['name'].nunique()}")

Extracted 521 total trait instances
Unique trait names: 154


In [72]:
# Trait name frequencies
trait_name_counts = traits_df['name'].value_counts()
print("\nTop 30 most common trait names:")
print(trait_name_counts.head(30).to_string())


Top 30 most common trait names:
name
Magic Resistance                31
Amphibious                      30
Legendary Resistance (3/Day)    23
Innate Spellcasting             20
Keen Smell                      18
Pack Tactics                    17
False Appearance                15
Keen Hearing and Smell          13
Shapechanger                    13
Magic Weapons                   12
Spider Climb                    12
Spellcasting                    12
Charge                          12
Swarm                           10
Water Breathing                  9
Hold Breath                      8
Devil's Sight                    8
Web Walker                       7
Keen Sight                       7
Sunlight Sensitivity             7
Web Sense                        5
Death Burst                      5
Stone Camouflage                 5
Trampling Charge                 5
Blood Frenzy                     5
Echolocation                     5
Regeneration                     5
Innate Spellcasti

In [73]:
traits_df[traits_df['name'] == 'Rampage']

,monster,cr,name,description,source
192,Giant Hyena,Unknown,Rampage,When the hyena reduces a creature to 0 hit poi...,Traits
221,Gnoll,Unknown,Rampage,When the gnoll reduces a creature to 0 hit poi...,Traits


In [74]:
# Normalize trait names (remove parenthetical variations)
def normalize_trait_name(name):
    """Remove parenthetical suffixes like (Recharge 5-6), (3/Day), etc."""
    # Remove recharge notation
    normalized = re.sub(r'\s*\(Recharge\s*\d+[-–]?\d*\)', '', name, flags=re.IGNORECASE)
    # Remove X/Day notation  
    normalized = re.sub(r'\s*\(\d+/Day\)', '', normalized, flags=re.IGNORECASE)
    # Remove other common parentheticals
    normalized = re.sub(r'\s*\([^)]*form[^)]*\)', '', normalized, flags=re.IGNORECASE)
    return normalized.strip()

traits_df['name_normalized'] = traits_df['name'].apply(normalize_trait_name)

normalized_counts = traits_df['name_normalized'].value_counts()
print("\nTop 30 normalized trait names:")
print(normalized_counts.head(30).to_string())


Top 30 normalized trait names:
name_normalized
Magic Resistance          31
Amphibious                30
Innate Spellcasting       24
Legendary Resistance      23
Keen Smell                18
Pack Tactics              17
False Appearance          16
Shapechanger              13
Charge                    13
Keen Hearing and Smell    13
Magic Weapons             12
Spellcasting              12
Spider Climb              12
Swarm                     10
Water Breathing            9
Hold Breath                8
Devil's Sight              8
Sunlight Sensitivity       7
Web Walker                 7
Keen Sight                 7
Death Burst                5
Stone Camouflage           5
Pounce                     5
Regeneration               5
Blood Frenzy               5
Echolocation               5
Web Sense                  5
Trampling Charge           5
Siege Monster              4
Standing Leap              4


## Section 2: Extract All Actions

In [75]:
def extract_actions(df):
    """Extract all actions from monsters into a flat dataframe."""
    all_actions = []
    
    for idx, row in df.iterrows():
        monster_name = row['Name']
        cr = row.get('CR', 'Unknown')
        
        actions_str = row.get('Actions', '')
        if pd.isna(actions_str) or actions_str == '' or actions_str == '—':
            continue
            
        try:
            actions = json.loads(actions_str)
            for action in actions:
                if isinstance(action, dict):
                    all_actions.append({
                        'monster': monster_name,
                        'cr': cr,
                        'name': action.get('Name', 'Unknown'),
                        'description': action.get('Desc', action.get('Description', '')),
                        'damage': action.get('Damage', ''),
                        'damage_2': action.get('Damage 2', ''),
                        'hit_bonus': action.get('Hit Bonus', ''),
                        'source': 'Actions'
                    })
        except (json.JSONDecodeError, TypeError):
            continue
    
    return pd.DataFrame(all_actions)

actions_df = extract_actions(monsters_df)
print(f"Extracted {len(actions_df)} total action instances")
print(f"Unique action names: {actions_df['name'].nunique()}")

Extracted 882 total action instances
Unique action names: 199


In [76]:
# Action name frequencies
action_name_counts = actions_df['name'].value_counts()
print("\nTop 30 most common action names:")
print(action_name_counts.head(30).to_string())


Top 30 most common action names:
name
Multiattack                            137
Bite                                   135
Claw                                    49
Tail                                    32
Claws                                   28
Frightful Presence                      21
Breath Weapons (Recharge 5-6)           20
Spear                                   15
Longsword                               14
Slam                                    14
Hooves                                  13
Beak                                    12
Change Shape                            12
Javelin                                 10
Shortsword                               9
Bites (swarm has half HP or less)        9
Bites (swarm has more than half HP)      9
Rock                                     9
Longbow                                  8
Talons                                   8
Gore                                     7
Fire Breath (Recharge 5-6)               6
Dagger         

In [77]:
actions_df.loc[805, 'description']

'If the target is a humanoid, it must succeed on a DC 11 Constitution saving throw or be cursed with wererat lycanthropy'

In [78]:
actions_df[actions_df['name'] == 'Rock']

,monster,cr,name,description,damage,damage_2,hit_bonus,source
145,Ape,Unknown,Rock,,1d6 + 3,,5,Actions
237,Cloud Giant,Unknown,Rock,,4d10 + 8,,12,Actions
330,Fire Giant,Unknown,Rock,,4d10 + 7,,11,Actions
337,Frost Giant,Unknown,Rock,,4d10 + 6,,9,Actions
353,Giant Ape,Unknown,Rock,,7d6 + 6,,9,Actions
459,Hill Giant,Unknown,Rock,,3d10 + 5,,8,Actions
702,Stone Giant,Unknown,Rock,"If the target is a creature, it must succeed o...",4d10 + 6,,9,Actions
708,Storm Giant,Unknown,Rock,,4d12 + 9,,14,Actions
752,Treant,Unknown,Rock,,4d10 + 6,,10,Actions


In [79]:
# === Weapon / Natural Weapon Classification for Action Name Consolidation ===

# Load weapon data from CSV to build melee/ranged lookup
weapons_ref_df = pd.read_csv(MAIN_DATA_DIR + '/dnd5e_weapons.csv')

weapon_category_map = {}  # lowercase weapon name -> "Melee Weapon" or "Ranged Weapon"

for _, row in weapons_ref_df.iterrows():
    raw_name = row['name'].strip()
    category = row['weapon_category']
    consolidated = 'Melee Weapon' if 'Melee' in category else 'Ranged Weapon'
    
    # Add canonical CSV name
    weapon_category_map[raw_name.lower()] = consolidated
    
    # Handle "Crossbow, light" -> also register as "light crossbow"
    if ',' in raw_name:
        parts = [p.strip() for p in raw_name.split(',')]
        reversed_name = ' '.join(reversed(parts)).lower()
        weapon_category_map[reversed_name] = consolidated

# Aliases for weapons not in CSV but found in monster action names
weapon_aliases = {
    'heavy crossbow': 'Ranged Weapon',
    'hand crossbow': 'Ranged Weapon',
    'light crossbow': 'Ranged Weapon',
    'pistol': 'Ranged Weapon',
    'harpoon': 'Ranged Weapon',
    'bone bow': 'Ranged Weapon',
    'sword': 'Melee Weapon',
    'blade': 'Melee Weapon',
    'hammer': 'Melee Weapon',
    'staff': 'Melee Weapon',
    'scythe': 'Melee Weapon',
    'fork': 'Melee Weapon',
    # My Mods
    'rock': 'Ranged Weapon',
}
for alias, cat in weapon_aliases.items():
    if alias not in weapon_category_map:
        weapon_category_map[alias] = cat

# Thrown melee weapons that can appear with (Ranged) suffix
THROWN_MELEE_WEAPONS = {'spear', 'javelin', 'dagger', 'handaxe', 'light hammer', 'trident'}

# Sort by length descending so "light crossbow" matches before "club"
WEAPON_NAMES_BY_LENGTH = sorted(weapon_category_map.keys(), key=len, reverse=True)

# Natural weapons: strict body-part attacks only
NATURAL_WEAPONS = {
    'bite', 'claw', 'claws', 'tail', 'slam', 'hooves', 'beak', 'talons',
    'gore', 'ram', 'sting', 'pseudopod', 'constrict', 'rend', 'tentacles',
    'stomp', 'tentacle', 'horns', 'pincer', 'tusk', 'tusks', 'crush',
    'horn', 'rake', 'snake hair', 'stinger', 'tail spike', 'tail stinger',
    'antennae', 'beard', 'bites', 'beaks',
    'tentacle slam', 'otherworldly slam', 'paralyzing tentacles',
    # My Adds
    'fist',

}

# Candidates for manual review (NOT auto-classified)
NATURAL_WEAPON_CANDIDATES = [
    'unarmed strike', 'rotting fist', 'fling',
    'blood drain', 'slash', 'rotting touch', 'withering touch',
    'poison jab', 'shock',
    # My removed:
    # 'rock', 'fist'
]


def normalize_action_name(name):
    """
    Normalize action names with weapon/natural weapon consolidation:
    1. Strip parentheticals (same as normalize_trait_name)
    2. Classify manufactured weapons -> "Melee Weapon" / "Ranged Weapon"
    3. Classify natural body-part weapons -> "Natural Weapon"
    4. Everything else keeps its stripped name as-is
    """
    # Detect (Ranged...) suffix BEFORE stripping parentheticals
    has_ranged_suffix = bool(re.search(r'\(Ranged[^)]*\)', name, flags=re.IGNORECASE))
    
    # Strip ALL parentheticals
    normalized = re.sub(r'\s*\(Recharge\s*\d+[-–]?\d*\)', '', name, flags=re.IGNORECASE)
    normalized = re.sub(r'\s*\(\d+/Day\)', '', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'\s*\([^)]*\)', '', normalized)
    normalized = normalized.strip()
    # Handle trailing period (e.g., "Bite.")
    normalized = normalized.rstrip('.')
    
    normalized_lower = normalized.lower()
    
    # Check exact match against natural weapons
    if normalized_lower in NATURAL_WEAPONS:
        return 'Natural Weapon'
    
    # Check exact match against manufactured weapons
    if normalized_lower in weapon_category_map:
        if has_ranged_suffix and normalized_lower in THROWN_MELEE_WEAPONS:
            return 'Ranged Weapon'
        return weapon_category_map[normalized_lower]
    
    # Substring match for monster-specific variants (e.g., "Insectile Rapier")
    for weapon_name in WEAPON_NAMES_BY_LENGTH:
        if weapon_name in normalized_lower:
            if has_ranged_suffix and weapon_name in THROWN_MELEE_WEAPONS:
                return 'Ranged Weapon'
            return weapon_category_map[weapon_name]
    
    # No match - return stripped name as-is
    return normalized


print(f"Weapon lookup: {len(weapon_category_map)} entries "
      f"({sum(1 for v in weapon_category_map.values() if v == 'Melee Weapon')} melee, "
      f"{sum(1 for v in weapon_category_map.values() if v == 'Ranged Weapon')} ranged)")
print(f"Natural weapons: {len(NATURAL_WEAPONS)} entries")
print(f"Candidates for manual review: {len(NATURAL_WEAPON_CANDIDATES)} entries")

Weapon lookup: 50 entries (34 melee, 16 ranged)
Natural weapons: 36 entries
Candidates for manual review: 9 entries


In [80]:
# Normalize action names with weapon/natural weapon consolidation
actions_df['name_normalized'] = actions_df['name'].apply(normalize_action_name)

action_normalized_counts = actions_df['name_normalized'].value_counts()
print("Top 30 normalized action names:")
print(action_normalized_counts.head(30).to_string())

# Consolidation summary
print(f"\nConsolidation summary:")
print(f"  'Natural Weapon' instances: {(actions_df['name_normalized'] == 'Natural Weapon').sum()}")
print(f"  'Melee Weapon' instances: {(actions_df['name_normalized'] == 'Melee Weapon').sum()}")
print(f"  'Ranged Weapon' instances: {(actions_df['name_normalized'] == 'Ranged Weapon').sum()}")
other_count = (~actions_df['name_normalized'].isin(['Melee Weapon', 'Ranged Weapon', 'Natural Weapon'])).sum()
print(f"  Other (unconsolidated): {other_count}")
print(f"  Unique normalized names: {actions_df['name_normalized'].nunique()}")

# Show candidates for manual review (not auto-classified)
candidate_actions = actions_df[actions_df['name_normalized'].str.lower().isin(NATURAL_WEAPON_CANDIDATES)]
if len(candidate_actions) > 0:
    print(f"\n--- Natural Weapon candidates for manual review ({len(candidate_actions)} instances) ---")
    candidate_counts = candidate_actions['name_normalized'].value_counts()
    for name, count in candidate_counts.items():
        print(f"  {name}: {count}")


Top 30 normalized action names:
name_normalized
Natural Weapon         386
Multiattack            142
Melee Weapon           123
Ranged Weapon           41
Frightful Presence      21
Breath Weapons          20
Change Shape            12
Fire Breath              8
Swallow                  5
Cold Breath              5
Poison Breath            5
Lightning Breath         5
Teleport                 5
Invisibility             5
Etherealness             5
Charm                    4
Healing Touch            4
Acid Breath              4
Hurl Flame               3
Draining Kiss            3
Life Drain               3
Engulf                   2
Ink Cloud                2
Touch                    2
Illusory Appearance      2
Steam Breath             2
Rotting Fist             2
Dreadful Glare           2
Web                      2
Enslave                  1

Consolidation summary:
  'Natural Weapon' instances: 386
  'Melee Weapon' instances: 123
  'Ranged Weapon' instances: 41
  Other (unconsolida

In [81]:
actions_df[actions_df['name'] == 'Breath Weapon']

,monster,cr,name,description,damage,damage_2,hit_bonus,source,name_normalized


In [82]:
actions_df[actions_df['monster'] == 'Sahuagin']

,monster,cr,name,description,damage,damage_2,hit_bonus,source,name_normalized
650,Sahuagin,Unknown,Multiattack,The sahuagin makes two melee attacks: one with...,,,,Actions,Multiattack
651,Sahuagin,Unknown,Bite,,1d4 + 1,,3,Actions,Natural Weapon
652,Sahuagin,Unknown,Claws,,1d4 + 1,,3,Actions,Natural Weapon
653,Sahuagin,Unknown,Spear,two handed,1d8 + 1,,3,Actions,Melee Weapon
654,Sahuagin,Unknown,Spear,,1d6 + 1,,3,Actions,Melee Weapon
655,Sahuagin,Unknown,Spear,,1d6 + 1,,3,Actions,Melee Weapon


## Section 3: Pattern Detection

Scan all descriptions for mechanical patterns.

In [83]:
# Define pattern categories with regex
PATTERN_DEFINITIONS = {
    # Saving throws
    'has_saving_throw': r'dc\s+\d+\s+(?:strength|dexterity|constitution|intelligence|wisdom|charisma)',
    'save_or_effect': r'(?:fails?|failed)\s+(?:the\s+)?(?:saving throw|save)',
    'half_damage_on_save': r'half\s+(?:as\s+much\s+)?damage\s+on\s+a\s+successful',
    'repeat_save': r'repeat(?:s|ing)?\s+the\s+saving\s+throw|can\s+repeat\s+the\s+save',
    
    # Conditions inflicted
    'inflicts_blinded': r'(?:is|becomes?|be)\s+blinded|blinded\s+(?:until|for)',
    'inflicts_charmed': r'(?:is|becomes?|be)\s+charmed|charmed\s+(?:until|for)',
    'inflicts_frightened': r'(?:is|becomes?|be)\s+frightened|frightened\s+(?:until|for)',
    'inflicts_grappled': r'(?:is|becomes?|be)\s+grappled|grappled\s+(?:until|by)',
    'inflicts_incapacitated': r'(?:is|becomes?|be)\s+incapacitated|incapacitated\s+(?:until|for)',
    'inflicts_paralyzed': r'(?:is|becomes?|be)\s+paralyzed|paralyzed\s+(?:until|for)',
    'inflicts_petrified': r'(?:is|becomes?|be)\s+petrified|petrified\s+(?:until|for)',
    'inflicts_poisoned': r'(?:is|becomes?|be)\s+poisoned|poisoned\s+(?:until|for)',
    'inflicts_prone': r'(?:is|becomes?|be)\s+knocked\s+prone|knocked\s+prone',
    'inflicts_restrained': r'(?:is|becomes?|be)\s+restrained|restrained\s+(?:until|by)',
    'inflicts_stunned': r'(?:is|becomes?|be)\s+stunned|stunned\s+(?:until|for)',
    'inflicts_unconscious': r'(?:is|becomes?|be)\s+unconscious|falls?\s+unconscious',
    
    # Control mechanics
    'escape_dc': r'escape\s+dc\s+\d+|dc\s+\d+\s+(?:to\s+)?escape',
    'cant_move': r"can(?:'t|not)\s+move|speed\s+(?:is|becomes)\s+0",
    'speed_reduction': r'speed\s+(?:is\s+)?reduced|reduces?\s+(?:the\s+)?(?:target\'?s?\s+)?speed',
    
    # Damage patterns
    'extra_damage': r'plus\s+\d+|extra\s+\d+d\d+|additional\s+\d+',
    'damage_on_start_turn': r'(?:start|beginning)\s+of\s+(?:its|the\s+target\'?s?)\s+turn.*damage',
    'damage_on_end_turn': r'end\s+of\s+(?:its|the\s+target\'?s?)\s+turn.*damage',
    'ongoing_damage': r'takes?\s+\d+.*damage\s+(?:at\s+the\s+)?(?:start|end|each)',
    
    # HP manipulation
    'reduces_hp_max': r'hit\s+point\s+maximum\s+(?:is\s+)?reduced|reduces?\s+(?:the\s+)?(?:target\'?s?\s+)?(?:hit\s+point\s+)?maximum',
    'regains_hp': r'regains?\s+\d+\s+hit\s+points?|regains?\s+hit\s+points?',
    'cant_regain_hp': r"can(?:'t|not)\s+regain\s+hit\s+points?",
    'heals_on_damage': r'regains?\s+(?:a\s+number\s+of\s+)?hit\s+points?\s+equal\s+to',
    
    # Defensive abilities  
    'damage_resistance': r'resistance\s+to\s+(?:bludgeoning|piercing|slashing|fire|cold|lightning|thunder|poison|acid|necrotic|radiant|force|psychic)',
    'damage_immunity': r'immun(?:e|ity)\s+to\s+(?:bludgeoning|piercing|slashing|fire|cold|lightning|thunder|poison|acid|necrotic|radiant|force|psychic)',
    'condition_immunity': r'immun(?:e|ity)\s+to\s+(?:the\s+)?(?:charmed|frightened|poisoned|paralyzed|petrified|stunned)',
    'magic_resistance': r'magic\s+resistance|advantage\s+on\s+saving\s+throws\s+against\s+spells',
    'legendary_resistance': r'legendary\s+resistance',
    'damage_reduction': r'damage\s+(?:is\s+)?reduced\s+by|reduces?\s+(?:the\s+)?damage',
    
    # Regeneration
    'regeneration': r'regenerat(?:es?|ion)|regains?\s+\d+\s+hit\s+points?\s+at\s+the\s+start',
    'regeneration_prevention': r'(?:doesn\'?t|does\s+not)\s+(?:function|work)|prevents?\s+(?:this\s+)?regeneration',
    
    # Movement abilities
    'teleport': r'teleport|magically\s+(?:appears?|moves?|transports?)',
    'incorporeal': r'incorporeal|move\s+through\s+(?:other\s+)?creatures?\s+and\s+objects?',
    'ethereal': r'ethereal|etherealness|enters?\s+the\s+ethereal\s+plane',
    'phasing': r'phase|phasing|passes?\s+through',
    
    # Shapechanging
    'shapechange': r'shapechange|polymorph|changes?\s+(?:its\s+)?shape|transforms?\s+into',
    'revert_form': r'reverts?\s+to\s+(?:its\s+)?(?:true\s+)?form|returns?\s+to\s+(?:its\s+)?original',
    
    # Attack modifiers
    'advantage_on_attack': r'advantage\s+on\s+(?:the\s+)?attack\s+roll|attack\s+rolls?\s+(?:have|has|with)\s+advantage',
    'pack_tactics': r'pack\s+tactics',
    'sneak_attack': r'sneak\s+attack',
    'surprise_attack': r'surpris(?:e|ed).*attack|attack.*surpris(?:e|ed)',
    
    # Stealth / Hide abilities
    'bonus_action_hide': r'(?:take|use)\s+the\s+(?:hide|disengage)\s+action\s+as\s+a\s+bonus\s+action|hide\s+action\s+as\s+a\s+bonus\s+action',
    'stealth_advantage': r'advantage\s+on\s+(?:dexterity\s+)?\(?stealth\)?\s+checks?|naturally\s+stealthy',
    
    # Recharge abilities
    'recharge': r'recharge\s+\d+[-–]?\d*',
    'breath_weapon': r'breath(?:es?)?\s+(?:weapon|fire|frost|lightning|acid|poison)',
    
    # Death/destruction effects
    'death_burst': r'death\s+burst|when.*dies.*creature',
    'explodes_on_death': r'explodes?|when.*(?:dies|destroyed).*damage',
    
    # Undead special
    'undead_fortitude': r'undead\s+fortitude|drop(?:s|ped)?\s+to\s+0\s+hit\s+points?.*constitution\s+saving',
    
    # Spellcasting
    'innate_spellcasting': r'innate\s+spellcasting|innately\s+cast',
    'spellcasting': r'spellcasting|\d+(?:st|nd|rd|th)[-\s]level\s+spellcaster',
    
    # Aura effects
    'aura': r'aura|within\s+\d+\s+feet\s+of\s+(?:the|it)',
    
    # Multiattack patterns
    'multiattack': r'multiattack|makes?\s+(?:two|three|four|five|six)\s+(?:melee\s+)?attacks?',
    
    # Swallow/engulf
    'swallow': r'swallow(?:s|ed)?|engulf(?:s|ed)?',
    
    # Charm/dominate
    'charm_effect': r'charm(?:s|ed)?\s+(?:the\s+)?(?:target|creature)|charmed\s+by',
    
    # Fear effect
    'fear_effect': r'frightful\s+presence|frighten(?:s|ed)?\s+(?:the\s+)?(?:target|creature)',
    
    # Gaze attacks
    'gaze_attack': r'gaze|looks?\s+(?:at|into).*eyes',
    
    # Life drain
    'life_drain': r'life\s+drain|drain(?:s|ed)?\s+(?:life|hit\s+points?)',
}

print(f"Defined {len(PATTERN_DEFINITIONS)} pattern categories")

Defined 61 pattern categories


In [84]:
def detect_patterns(text, patterns=PATTERN_DEFINITIONS):
    """Detect which patterns are present in text."""
    if pd.isna(text) or text == '':
        return []
    
    text_lower = str(text).lower()
    found = []
    
    for pattern_name, regex in patterns.items():
        if re.search(regex, text_lower):
            found.append(pattern_name)
    
    return found

# Apply pattern detection to traits
traits_df['patterns'] = traits_df['description'].apply(detect_patterns)
traits_df['pattern_count'] = traits_df['patterns'].apply(len)

print(f"Traits with at least one pattern: {(traits_df['pattern_count'] > 0).sum()}")
print(f"Traits with no patterns: {(traits_df['pattern_count'] == 0).sum()}")

Traits with at least one pattern: 209
Traits with no patterns: 312


In [85]:
# Apply pattern detection to actions
actions_df['patterns'] = actions_df['description'].apply(detect_patterns)
actions_df['pattern_count'] = actions_df['patterns'].apply(len)

print(f"Actions with at least one pattern: {(actions_df['pattern_count'] > 0).sum()}")
print(f"Actions with no patterns: {(actions_df['pattern_count'] == 0).sum()}")

Actions with at least one pattern: 356
Actions with no patterns: 526


In [86]:
# Core patterns from notebooks/helper_files/parsers.py
# These are the patterns used in the main feature engineering pipeline

CORE_PATTERN_DEFINITIONS = {
    # From has_advantage_condition()
    'core_pack_tactics': r'pack tactics',
    'core_blood_frenzy': r'blood frenzy',
    'core_reckless': r'reckless',
    'core_ambusher': r'ambusher',
    'core_assassinate': r'assassinate',
    'core_grappler': r'grappler',
    'core_has_advantage_attack': r'has advantage on.{0,20}attack roll',
    'core_have_advantage_attack': r'have advantage on.{0,20}attack roll',
    'core_advantage_against': r'advantage on attack rolls against',
    
    # From has_disadvantage_condition()
    'core_sunlight_sensitivity': r'sunlight sensitivity',
    'core_sunlight_weakness': r'sunlight weakness',
    'core_light_sensitivity': r'light sensitivity',
    
    # From has_attackers_advantage()
    'core_attackers_advantage_1': r'attack rolls? against.{0,20}have advantage',
    'core_attackers_advantage_2': r'attacks? against.{0,20}has advantage',
    
    # From parse_charge_bonus_attack() - charge/pounce/rampage patterns
    'core_charge': r'charge',
    'core_pounce': r'pounce',
    'core_rampage': r'rampage',
    'core_trampling': r'trampling',
    
    # From extract_spellcaster_level()
    'core_spellcaster_level': r'(\d+)(?:st|nd|rd|th)[-\s]level\s+spellcaster',
    'core_casts_as_level': r'casts?\s+spells?\s+as\s+a?\s*(\d+)(?:st|nd|rd|th)[-\s]level',
    
    # Legendary actions patterns from parse_legendary_actions()
    'core_legendary_actions': r'(\d+)\s+legendary\s+actions?',
    
    # Multiattack patterns from parse_dpr_from_json()
    'core_multiattack_two': r'(\w+)\s+two\s+attacks?',
    'core_multiattack_three': r'(\w+)\s+three\s+attacks?',
    'core_multiattack_with': r'with\s+(?:its\s+)?(\w+)',
    
    # Conditional damage patterns from parse_dpr_from_json()
    'core_taking_damage': r'taking\s+(\d+)\s*\(([^)]+)\)\s*(?:\w+\s+)?damage',
    'core_takes_damage': r'takes\s+(\d+)\s*\(([^)]+)\)\s*(?:\w+\s+)?damage',
    'core_plus_damage': r'plus\s+(\d+)\s*\(([^)]+)\)\s*(?:\w+\s+)?damage',
    
    # Prone infliction from various parsers
    'core_knocked_prone': r'knocked\s+prone',
    'core_falls_prone': r'falls?\s+prone',
}

print(f"Defined {len(CORE_PATTERN_DEFINITIONS)} core patterns from parsers.py")

Defined 29 core patterns from parsers.py


In [87]:
# Apply core patterns to traits and actions
def detect_core_patterns(text):
    """Detect core patterns from parsers.py in text."""
    if pd.isna(text) or text == '':
        return []
    
    text_lower = str(text).lower()
    found = []
    
    for pattern_name, regex in CORE_PATTERN_DEFINITIONS.items():
        if re.search(regex, text_lower):
            found.append(pattern_name)
    
    return found

# Apply to traits
traits_df['patterns_core'] = traits_df['description'].apply(detect_core_patterns)
traits_df['pattern_count_core'] = traits_df['patterns_core'].apply(len)
traits_df['patterns_str_core'] = traits_df['patterns_core'].apply(lambda x: '|'.join(x) if x else '')

# Apply to actions
actions_df['patterns_core'] = actions_df['description'].apply(detect_core_patterns)
actions_df['pattern_count_core'] = actions_df['patterns_core'].apply(len)
actions_df['patterns_str_core'] = actions_df['patterns_core'].apply(lambda x: '|'.join(x) if x else '')

print(f"Traits with core patterns: {(traits_df['pattern_count_core'] > 0).sum()}")
print(f"Actions with core patterns: {(actions_df['pattern_count_core'] > 0).sum()}")

Traits with core patterns: 125
Actions with core patterns: 222


In [88]:
# Create combined pattern columns (notebook patterns + core patterns)
def combine_patterns(row):
    """Combine patterns from both sources, removing duplicates."""
    notebook_patterns = row['patterns'] if isinstance(row['patterns'], list) else []
    core_patterns = row['patterns_core'] if isinstance(row['patterns_core'], list) else []
    # Combine and dedupe
    combined = list(set(notebook_patterns + core_patterns))
    return combined

# Apply to traits
traits_df['patterns_any'] = traits_df.apply(combine_patterns, axis=1)
traits_df['pattern_count_any'] = traits_df['patterns_any'].apply(len)
traits_df['patterns_str_any'] = traits_df['patterns_any'].apply(lambda x: '|'.join(sorted(x)) if x else '')

# Apply to actions
actions_df['patterns_any'] = actions_df.apply(combine_patterns, axis=1)
actions_df['pattern_count_any'] = actions_df['patterns_any'].apply(len)
actions_df['patterns_str_any'] = actions_df['patterns_any'].apply(lambda x: '|'.join(sorted(x)) if x else '')

print("Combined pattern coverage:")
print(f"  Traits - notebook only: {(traits_df['pattern_count'] > 0).sum()}, "
      f"core only: {(traits_df['pattern_count_core'] > 0).sum()}, "
      f"any: {(traits_df['pattern_count_any'] > 0).sum()}")
print(f"  Actions - notebook only: {(actions_df['pattern_count'] > 0).sum()}, "
      f"core only: {(actions_df['pattern_count_core'] > 0).sum()}, "
      f"any: {(actions_df['pattern_count_any'] > 0).sum()}")

# Show traits/actions that are ONLY captured by core patterns (not notebook patterns)
core_only_traits = traits_df[(traits_df['pattern_count_core'] > 0) & (traits_df['pattern_count'] == 0)]
core_only_actions = actions_df[(actions_df['pattern_count_core'] > 0) & (actions_df['pattern_count'] == 0)]
print(f"\nTraits captured ONLY by core patterns: {len(core_only_traits)}")
print(f"Actions captured ONLY by core patterns: {len(core_only_actions)}")

Combined pattern coverage:
  Traits - notebook only: 209, core only: 125, any: 262
  Actions - notebook only: 356, core only: 222, any: 372

Traits captured ONLY by core patterns: 53
Actions captured ONLY by core patterns: 16


In [89]:
# Pattern frequency across all traits and actions
all_patterns = []
for patterns in traits_df['patterns']:
    all_patterns.extend(patterns)
for patterns in actions_df['patterns']:
    all_patterns.extend(patterns)

pattern_counts = Counter(all_patterns)
pattern_freq_df = pd.DataFrame([
    {'pattern': p, 'count': c} 
    for p, c in pattern_counts.most_common()
])

print("\nPattern frequencies (all traits + actions):")
print(pattern_freq_df.to_string())


Pattern frequencies (all traits + actions):
                    pattern  count
0          has_saving_throw    232
1               multiattack     99
2            save_or_effect     91
3       half_damage_on_save     79
4               repeat_save     65
5                      aura     63
6               fear_effect     45
7       inflicts_restrained     37
8            inflicts_prone     37
9         inflicts_grappled     37
10             spellcasting     34
11         magic_resistance     31
12      inflicts_frightened     31
13                escape_dc     31
14              shapechange     26
15        inflicts_poisoned     26
16               regains_hp     25
17      innate_spellcasting     24
18              revert_form     24
19      advantage_on_attack     21
20            breath_weapon     20
21       inflicts_paralyzed     18
22           ongoing_damage     14
23           cant_regain_hp     14
24                  swallow     13
25         inflicts_blinded     12
26        

## Section 4: Identify Unmatched Traits/Actions

Find traits and actions that don't match any patterns (potential gaps).

In [90]:
# Traits with no patterns detected (using combined notebook + core patterns)
unmatched_traits = traits_df[traits_df['pattern_count_any'] == 0].copy()
print(f"\nUnmatched traits (no notebook OR core patterns): {len(unmatched_traits)}")

# Compare with notebook-only patterns
unmatched_notebook_only = traits_df[traits_df['pattern_count'] == 0]
print(f"(For comparison, unmatched with notebook patterns only: {len(unmatched_notebook_only)})")

# Group by normalized name
unmatched_trait_names = unmatched_traits['name_normalized'].value_counts()
print("\nTop 30 unmatched trait names:")
print(unmatched_trait_names.head(30).to_string())


Unmatched traits (no notebook OR core patterns): 259
(For comparison, unmatched with notebook patterns only: 312)

Top 30 unmatched trait names:
name_normalized
Amphibious                                           30
Legendary Resistance                                 23
Keen Smell                                           18
False Appearance                                     16
Keen Hearing and Smell                               13
Magic Weapons                                        12
Spider Climb                                         12
Water Breathing                                       9
Hold Breath                                           8
Devil's Sight                                         8
Web Walker                                            7
Keen Sight                                            7
Echolocation                                          5
Siege Monster                                         4
Amorphous                                             

In [91]:
# Actions with no patterns detected (using combined notebook + core patterns, excluding basic attacks)
unmatched_actions = actions_df[actions_df['pattern_count_any'] == 0].copy()

# Filter out consolidated weapon/natural weapon categories and basic attack patterns
CONSOLIDATED_CATEGORIES = {'melee weapon', 'ranged weapon', 'natural weapon'}

basic_attack_patterns = ['bite', 'claw', 'slam', 'fist', 'sword', 'dagger', 'mace', 'spear', 
                         'greataxe', 'greatsword', 'longsword', 'shortsword', 'scimitar',
                         'club', 'quarterstaff', 'javelin', 'handaxe', 'light crossbow',
                         'longbow', 'shortbow', 'crossbow', 'pike', 'halberd', 'morningstar',
                         'war pick', 'flail', 'glaive', 'trident', 'lance', 'whip', 'net',
                         'talon', 'tentacle', 'tail', 'hoof', 'horn', 'tusk', 'gore', 'sting',
                         'beak', 'pincers', 'pseudopod', 'rock', 'touch', 'constrict']

def is_basic_attack(row):
    # Exclude consolidated categories
    if row['name_normalized'].lower() in CONSOLIDATED_CATEGORIES:
        return True
    # Also check original name for any remaining basic attack substring matches
    name_lower = str(row['name']).lower()
    return any(pattern in name_lower for pattern in basic_attack_patterns)

unmatched_special_actions = unmatched_actions[~unmatched_actions.apply(is_basic_attack, axis=1)]
print(f"\nUnmatched actions (excluding basic/weapon/natural attacks, using combined patterns): {len(unmatched_special_actions)}")

# Compare with notebook-only patterns
unmatched_notebook_only = actions_df[actions_df['pattern_count'] == 0]
unmatched_notebook_special = unmatched_notebook_only[~unmatched_notebook_only.apply(is_basic_attack, axis=1)]
print(f"(For comparison, unmatched with notebook patterns only: {len(unmatched_notebook_special)})")

unmatched_action_names = unmatched_special_actions['name_normalized'].value_counts()
print(f"\nTop 30 unmatched special action names:")
print(unmatched_action_names.head(30).to_string())



Unmatched actions (excluding basic/weapon/natural attacks, using combined patterns): 48
(For comparison, unmatched with notebook patterns only: 62)

Top 30 unmatched special action names:
name_normalized
Multiattack              36
Hurl Flame                3
Ink Cloud                 2
Haste                     1
Read Thoughts             1
Strength Drain            1
Blood Drain               1
Children of the Night     1
Shock                     1
Invisibility              1


In [92]:
actions_df[actions_df['name'] == 'Breath Weapon']

,monster,cr,name,description,damage,damage_2,hit_bonus,source,name_normalized,patterns,pattern_count,patterns_core,pattern_count_core,patterns_str_core,patterns_any,pattern_count_any,patterns_str_any


In [93]:
traits_df[traits_df['name_normalized'] == 'aura']

,monster,cr,name,description,source,name_normalized,patterns,pattern_count,patterns_core,pattern_count_core,patterns_str_core,patterns_any,pattern_count_any,patterns_str_any


In [94]:
# [for xx in monsters_df[monsters_df['Name'] == 'Warhorse']['Actions'].iloc[0] ]

## Section 5: Analyze Specific High-Priority Patterns

In [95]:
# Find all traits/actions with HP max reduction
hp_max_traits = traits_df[traits_df['patterns'].apply(lambda x: 'reduces_hp_max' in x)]
hp_max_actions = actions_df[actions_df['patterns'].apply(lambda x: 'reduces_hp_max' in x)]

print("=" * 60)
print("TRAITS/ACTIONS WITH HP MAX REDUCTION")
print("=" * 60)
print(f"\nTraits: {len(hp_max_traits)}")
for _, row in hp_max_traits.iterrows():
    print(f"  [{row['monster']}] {row['name']}")
    
print(f"\nActions: {len(hp_max_actions)}")
for _, row in hp_max_actions.head(20).iterrows():
    print(f"  [{row['monster']}] {row['name']}")

TRAITS/ACTIONS WITH HP MAX REDUCTION

Traits: 0

Actions: 12
  [Clay Golem] Slam
  [Incubus] Draining Kiss
  [Mummy] Rotting Fist
  [Mummy Lord] Rotting Fist
  [Night Hag] Nightmare Haunting (1/Day)
  [Specter] Life Drain
  [Succubus] Draining Kiss
  [Succubus (Incubus)] Draining Kiss
  [Vampire] Bite (Bat or Vampire Form Only)
  [Vampire Spawn] Bite
  [Wight] Life Drain
  [Wraith] Life Drain


In [96]:
# Find all creatures with shapechange
shapechange_traits = traits_df[traits_df['patterns'].apply(lambda x: 'shapechange' in x)]
shapechange_actions = actions_df[actions_df['patterns'].apply(lambda x: 'shapechange' in x)]

print("=" * 60)
print("CREATURES WITH SHAPECHANGE")
print("=" * 60)
all_shapeshifters = set(shapechange_traits['monster'].tolist() + shapechange_actions['monster'].tolist())
print(f"\nTotal creatures: {len(all_shapeshifters)}")
for creature in sorted(all_shapeshifters):
    print(f"  {creature}")

CREATURES WITH SHAPECHANGE

Total creatures: 25
  Adult Bronze Dragon
  Adult Gold Dragon
  Adult Silver Dragon
  Ancient Brass Dragon
  Ancient Bronze Dragon
  Ancient Copper Dragon
  Ancient Gold Dragon
  Ancient Silver Dragon
  Couatl
  Deva
  Doppelganger
  Imp
  Incubus
  Mimic
  Night Hag
  Oni
  Quasit
  Succubus
  Succubus (Incubus)
  Vampire
  Werebear
  Wereboar
  Wererat
  Weretiger
  Werewolf


In [97]:
# Find all death burst / explodes on death
death_burst_traits = traits_df[traits_df['patterns'].apply(lambda x: 'death_burst' in x or 'explodes_on_death' in x)]

print("=" * 60)
print("CREATURES WITH DEATH BURST / EXPLODES ON DEATH")
print("=" * 60)
for _, row in death_burst_traits.iterrows():
    print(f"\n[{row['monster']}] {row['name']}:")
    print(f"  {row['description'][:200]}..." if len(str(row['description'])) > 200 else f"  {row['description']}")

CREATURES WITH DEATH BURST / EXPLODES ON DEATH

[Balor] Death Throes:
  When the balor dies, it explodes, and each creature within 30 feet of it must make a DC 20 Dexterity saving throw, taking 70 (20d6) fire damage on a failed save, or half as much damage on a successful...

[Dust Mephit] Death Burst:
  When the mephit dies, it explodes in a burst of dust. Each creature within 5 ft. of it must then succeed on a DC 10 Constitution saving throw or be blinded for 1 minute. A blinded creature can repeat ...

[Flesh Golem] Berserk:
  Whenever the golem starts its turn with 40 hit points or fewer, roll a d6. On a 6, the golem goes berserk. On each of its turns while berserk, the golem attacks the nearest creature it can see. If no ...

[Hydra] Multiple Heads:
  The hydra has five heads. While it has more than one head, the hydra has advantage on saving throws against being blinded, charmed, deafened, frightened, stunned, and knocked unconscious. Whenever the...

[Ice Mephit] Death Burst:
  

In [98]:
# Find undead fortitude
fortitude_traits = traits_df[traits_df['patterns'].apply(lambda x: 'undead_fortitude' in x)]

print("=" * 60)
print("CREATURES WITH UNDEAD FORTITUDE")
print("=" * 60)
for _, row in fortitude_traits.iterrows():
    print(f"  [{row['monster']}] {row['name']}")

CREATURES WITH UNDEAD FORTITUDE


In [99]:
# Find ethereal / incorporeal movement
ethereal_traits = traits_df[traits_df['patterns'].apply(lambda x: 'ethereal' in x or 'incorporeal' in x)]

print("=" * 60)
print("CREATURES WITH ETHEREAL/INCORPOREAL MOVEMENT")
print("=" * 60)
for _, row in ethereal_traits.iterrows():
    print(f"  [{row['monster']}] {row['name']}")

CREATURES WITH ETHEREAL/INCORPOREAL MOVEMENT
  [Ghost] Ethereal Sight
  [Ghost] Incorporeal Movement
  [Night Hag] Night Hag Items
  [Phase Spider] Ethereal Jaunt
  [Specter] Incorporeal Movement
  [Will-o'-Wisp] Incorporeal Movement
  [Wraith] Incorporeal Movement


## Section 6: Save Outputs

In [100]:
# Convert patterns list to string for parquet compatibility
traits_df['patterns_str'] = traits_df['patterns'].apply(lambda x: '|'.join(x) if x else '')
actions_df['patterns_str'] = actions_df['patterns'].apply(lambda x: '|'.join(x) if x else '')

# Save trait catalog
traits_df.to_parquet(LOCAL_DATA_DIR + '/trait_catalog.parquet', index=False)
print(f"Saved trait catalog: {len(traits_df)} rows")

# Save action catalog
actions_df.to_parquet(LOCAL_DATA_DIR + '/action_catalog.parquet', index=False)
print(f"Saved action catalog: {len(actions_df)} rows")

# Save pattern frequencies
pattern_freq_df.to_csv(LOCAL_DATA_DIR + '/pattern_frequencies.csv', index=False)
print(f"Saved pattern frequencies: {len(pattern_freq_df)} patterns")

Saved trait catalog: 521 rows
Saved action catalog: 882 rows
Saved pattern frequencies: 53 patterns


In [101]:
# Summary statistics
print("\n" + "=" * 60)
print("PHASE 1 SUMMARY")
print("=" * 60)
print(f"\nTotal monsters: {len(monsters_df)}")
print(f"Total trait instances: {len(traits_df)}")
print(f"Unique trait names: {traits_df['name'].nunique()}")
print(f"Unique normalized trait names: {traits_df['name_normalized'].nunique()}")
print(f"\nTotal action instances: {len(actions_df)}")
print(f"Unique action names: {actions_df['name'].nunique()}")
print(f"Unique normalized action names: {actions_df['name_normalized'].nunique()}")
print(f"\nPatterns defined: {len(PATTERN_DEFINITIONS)}")
print(f"Patterns detected at least once: {len(pattern_freq_df)}")
print(f"\nTraits with patterns: {(traits_df['pattern_count'] > 0).sum()} ({100*(traits_df['pattern_count'] > 0).mean():.1f}%)")
print(f"Actions with patterns: {(actions_df['pattern_count'] > 0).sum()} ({100*(actions_df['pattern_count'] > 0).mean():.1f}%)")


PHASE 1 SUMMARY

Total monsters: 324
Total trait instances: 521
Unique trait names: 154
Unique normalized trait names: 150

Total action instances: 882
Unique action names: 199
Unique normalized action names: 86

Patterns defined: 61
Patterns detected at least once: 53

Traits with patterns: 209 (40.1%)
Actions with patterns: 356 (40.4%)


In [102]:
PATTERN_DEFINITIONS.keys()

dict_keys(['has_saving_throw', 'save_or_effect', 'half_damage_on_save', 'repeat_save', 'inflicts_blinded', 'inflicts_charmed', 'inflicts_frightened', 'inflicts_grappled', 'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified', 'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained', 'inflicts_stunned', 'inflicts_unconscious', 'escape_dc', 'cant_move', 'speed_reduction', 'extra_damage', 'damage_on_start_turn', 'damage_on_end_turn', 'ongoing_damage', 'reduces_hp_max', 'regains_hp', 'cant_regain_hp', 'heals_on_damage', 'damage_resistance', 'damage_immunity', 'condition_immunity', 'magic_resistance', 'legendary_resistance', 'damage_reduction', 'regeneration', 'regeneration_prevention', 'teleport', 'incorporeal', 'ethereal', 'phasing', 'shapechange', 'revert_form', 'advantage_on_attack', 'pack_tactics', 'sneak_attack', 'surprise_attack', 'bonus_action_hide', 'stealth_advantage', 'recharge', 'breath_weapon', 'death_burst', 'explodes_on_death', 'undead_fortitude', 'innate_s

## Section 7: Quick Reference - All Unique Trait Names

For manual review and categorization.

In [103]:
# Export all unique trait names with counts and pattern info (both notebook and core patterns)
trait_summary = traits_df.groupby('name_normalized').agg({
    'monster': 'count',
    'patterns_str': lambda x: '|'.join(set('|'.join(x).split('|')) - {''}),
    'patterns_str_core': lambda x: '|'.join(set('|'.join(x).split('|')) - {''}),
    'patterns_str_any': lambda x: '|'.join(set('|'.join(x).split('|')) - {''})
}).reset_index()
trait_summary.columns = ['trait_name', 'count', 'patterns_notebook', 'patterns_core', 'patterns_any']
trait_summary = trait_summary.sort_values('count', ascending=False)

trait_summary.to_csv(LOCAL_DATA_DIR + '/trait_summary.csv', index=False)
print(f"Saved trait summary: {len(trait_summary)} unique traits")
print("\nTop 50 traits by frequency:")
print(trait_summary.head(50).to_string())

Saved trait summary: 150 unique traits

Top 50 traits by frequency:
                                            trait_name  count                                                                                               patterns_notebook                                 patterns_core                                                                                                                       patterns_any
84                                    Magic Resistance     31                                                                                                magic_resistance                                                                                                                                                                 magic_resistance
7                                           Amphibious     30                                                                                                                                                                         

In [104]:
# Export all unique action names with counts (both notebook and core patterns)
action_summary = actions_df.groupby('name_normalized').agg({
    'monster': 'count',
    'patterns_str': lambda x: '|'.join(set('|'.join(x).split('|')) - {''}),
    'patterns_str_core': lambda x: '|'.join(set('|'.join(x).split('|')) - {''}),
    'patterns_str_any': lambda x: '|'.join(set('|'.join(x).split('|')) - {''})
}).reset_index()
action_summary.columns = ['action_name', 'count', 'patterns_notebook', 'patterns_core', 'patterns_any']
action_summary = action_summary.sort_values('count', ascending=False)

action_summary.to_csv(LOCAL_DATA_DIR + '/action_summary.csv', index=False)
print(f"Saved action summary: {len(action_summary)} unique actions")
print("\nTop 50 actions by frequency:")
print(action_summary.head(50).to_string())

Saved action summary: 86 unique actions

Top 50 actions by frequency:
              action_name  count                                                                                                                                                                                                                                                                                                                                       patterns_notebook                                                                                                             patterns_core                                                                                                                                                                                                                                                                                                                                                                                                                                                

## Section 8: Multiattack Pattern Analysis

Based on findings from `multiattack_parser_issues.md` and `parse_dpr_from_json` in `parsers.py`.

In [105]:
# Extract all multiattack descriptions from actions
multiattack_df = actions_df[actions_df['name'].str.lower() == 'multiattack'].copy()
print(f"Total creatures with Multiattack: {len(multiattack_df)}")
print(f"\nSample multiattack descriptions:")
for _, row in multiattack_df.head(5).iterrows():
    print(f"  [{row['monster']}]: {row['description'][:100]}...")

Total creatures with Multiattack: 137

Sample multiattack descriptions:
  [Aboleth]: The aboleth makes three tentacle attacks....
  [Adult Black Dragon]: The dragon can use its Frightful Presence. It then makes three attacks: one with its bite and two wi...
  [Adult Blue Dragon]: The dragon can use its Frightful Presence. It then makes three attacks: one with its bite and two wi...
  [Adult Brass Dragon]: The dragon can use its Frightful Presence. It then makes three attacks: one with its bite and two wi...
  [Adult Bronze Dragon]: The dragon can use its Frightful Presence. It then makes three attacks: one with its bite and two wi...


In [106]:
# Multiattack-specific patterns (from parse_dpr_from_json and multiattack_parser_issues)
MULTIATTACK_PATTERNS = {
    # Attack count patterns
    'count_two': r'makes?\s+two\s+(?:melee\s+)?attacks?',
    'count_three': r'makes?\s+three\s+(?:melee\s+)?attacks?',
    'count_four': r'makes?\s+four\s+(?:melee\s+)?attacks?',
    'count_variable': r'as many\s+\w+\s+attacks?\s+as\s+it\s+has',

    # OR alternative patterns (problematic - parser sums instead of max)
    'or_alternative': r'\bor\b(?!.*saving)',  # "or" not followed by "saving throw"
    'either_or': r'either\s+\w+\s+or',

    # Specific attack references
    'with_its': r'with\s+(?:its|her|his)\s+\w+',
    'one_with': r'one\s+with\s+(?:its|her|his)\s+\w+',
    'two_with': r'two\s+with\s+(?:its|her|his)\s+\w+',

    # Constraint patterns
    'only_one_can_be': r'only\s+one\s+(?:of\s+which\s+)?can\s+be',
    'at_most_one': r'at\s+most\s+one',
    'no_more_than': r'no\s+more\s+than\s+one',

    # Conditional patterns
    'if_condition': r'\bif\b.*(?:attack|hit|grappl)',
    'can_use_in_place': r'can\s+use\s+(?:its|her|his)\s+\w+\s+in\s+place\s+of',

    # Form-dependent patterns
    'in_form': r'in\s+(?:\w+\s+)?form',
    'while_in': r'while\s+in\s+(?:\w+\s+)?form',

    # Generic attack patterns (hard to parse)
    'melee_attacks': r'\d+\s+melee\s+attacks?(?!\s+with)',
    'ranged_attacks': r'\d+\s+ranged\s+attacks?(?!\s+with)',

    # Weapon-specific patterns
    'claw_attacks': r'(?:two|three|four)\s+(?:\w+\s+)?claw\s+attacks?',
    'bite_attack': r'one\s+(?:\w+\s+)?bite\s+attack',
    'weapon_attack': r'(?:two|three)\s+(?:\w+\s+)?(?:longsword|shortsword|greatsword|greataxe)\s+attacks?',
}

print(f"Defined {len(MULTIATTACK_PATTERNS)} multiattack-specific patterns")

Defined 21 multiattack-specific patterns


In [107]:
def detect_multiattack_patterns(desc):
    """Detect multiattack-specific patterns in description."""
    if pd.isna(desc) or desc == '':
        return []

    desc_lower = str(desc).lower()
    found = []

    for pattern_name, regex in MULTIATTACK_PATTERNS.items():
        if re.search(regex, desc_lower):
            found.append(pattern_name)

    return found

# Apply to multiattack descriptions
multiattack_df['ma_patterns'] = multiattack_df['description'].apply(detect_multiattack_patterns)
multiattack_df['ma_pattern_count'] = multiattack_df['ma_patterns'].apply(len)

print(f"Multiattacks with specific patterns: {(multiattack_df['ma_pattern_count'] > 0).sum()}")
print(f"Multiattacks with no detected patterns: {(multiattack_df['ma_pattern_count'] == 0).sum()}")

Multiattacks with specific patterns: 117
Multiattacks with no detected patterns: 20


In [108]:
multiattack_df[multiattack_df['ma_pattern_count'] == 0].head()

,monster,cr,name,description,damage,damage_2,hit_bonus,source,name_normalized,patterns,pattern_count,patterns_core,pattern_count_core,patterns_str_core,patterns_any,pattern_count_any,patterns_str_any,patterns_str,ma_patterns,ma_pattern_count
0,Aboleth,Unknown,Multiattack,The aboleth makes three tentacle attacks.,,,,Actions,Multiattack,[],0,[],0,,[],0,,,[],0
68,Air Elemental,Unknown,Multiattack,The elemental makes two slam attacks.,,,,Actions,Multiattack,[],0,[],0,,[],0,,,[],0
143,Ape,Unknown,Multiattack,The ape makes two fist attacks.,,,,Actions,Multiattack,[],0,[],0,,[],0,,,[],0
227,Clay Golem,Unknown,Multiattack,The golem makes two slam attacks.,,,,Actions,Multiattack,[],0,[],0,,[],0,,,[],0
235,Cloud Giant,Unknown,Multiattack,The giant makes two morningstar attacks.,,,,Actions,Multiattack,[],0,[],0,,[],0,,,[],0


In [109]:
# Count multiattack pattern frequencies
ma_patterns_list = []
for patterns in multiattack_df['ma_patterns']:
    ma_patterns_list.extend(patterns)

ma_pattern_counts = Counter(ma_patterns_list)
ma_pattern_freq_df = pd.DataFrame([
    {'pattern': p, 'count': c}
    for p, c in ma_pattern_counts.most_common()
])

print("\nMultiattack Pattern Frequencies:")
print(ma_pattern_freq_df.to_string())


Multiattack Pattern Frequencies:
             pattern  count
0           with_its     85
1           one_with     77
2        count_three     46
3           two_with     45
4          count_two     45
5     or_alternative     16
6      weapon_attack      9
7       claw_attacks      6
8       if_condition      6
9         count_four      3
10       bite_attack      2
11    count_variable      1
12   only_one_can_be      1
13           in_form      1
14  can_use_in_place      1


In [110]:
# Flag potentially problematic multiattacks (from parser issues analysis)
PROBLEMATIC_PATTERNS = ['or_alternative', 'either_or', 'count_variable',
                        'only_one_can_be', 'if_condition', 'in_form',
                        'melee_attacks', 'ranged_attacks']

def has_problematic_pattern(patterns):
    """Check if multiattack has patterns known to cause parsing issues."""
    return any(p in PROBLEMATIC_PATTERNS for p in patterns)

multiattack_df['has_parsing_issue'] = multiattack_df['ma_patterns'].apply(has_problematic_pattern)

print(f"\nPotentially problematic multiattacks: {multiattack_df['has_parsing_issue'].sum()}")
print(f"Percentage with issues: {100 * multiattack_df['has_parsing_issue'].mean():.1f}%")

# List problematic creatures
print("\nCreatures with potentially problematic multiattack patterns:")
problematic = multiattack_df[multiattack_df['has_parsing_issue']]
for _, row in problematic.iterrows():
    print(f"  [{row['monster']}] Patterns: {row['ma_patterns']}")
    print(f"    Desc: {row['description'][:80]}...")


Potentially problematic multiattacks: 23
Percentage with issues: 16.8%

Creatures with potentially problematic multiattack patterns:
  [Bandit Captain] Patterns: ['count_three', 'or_alternative', 'with_its', 'one_with', 'two_with']
    Desc: The captain makes three melee attacks: two with its scimitar and one with its da...
  [Centaur] Patterns: ['count_two', 'or_alternative', 'with_its', 'one_with', 'two_with']
    Desc: The centaur makes two attacks: one with its pike and one with its hooves or two ...
  [Chimera] Patterns: ['count_three', 'or_alternative', 'with_its', 'one_with']
    Desc: The chimera makes three attacks: one with its bite, one with its horns, and one ...
  [Chuul] Patterns: ['if_condition']
    Desc: The chuul makes two pincer attacks. If the chuul is grappling a creature, the ch...
  [Drider] Patterns: ['count_three', 'or_alternative', 'with_its']
    Desc: The drider makes three attacks, either with its longsword or its longbow. It can...
  [Efreeti] Patterns: [

In [111]:
# Categorize multiattacks by complexity
def categorize_multiattack(row):
    """Categorize multiattack by parsing complexity."""
    patterns = row['ma_patterns']
    desc = str(row['description']).lower()

    # Variable count (hardest)
    if 'count_variable' in patterns:
        return 'variable_count'

    # OR alternatives
    if 'or_alternative' in patterns or 'either_or' in patterns:
        return 'or_alternative'

    # Form-dependent
    if 'in_form' in patterns or 'while_in' in patterns:
        return 'form_dependent'

    # Conditional
    if 'if_condition' in patterns:
        return 'conditional'

    # Constrained ("only one can be")
    if 'only_one_can_be' in patterns or 'at_most_one' in patterns:
        return 'constrained'

    # Generic (no specific weapons)
    if 'melee_attacks' in patterns or 'ranged_attacks' in patterns:
        return 'generic'

    # Simple (specific attack references)
    if any(p in patterns for p in ['with_its', 'one_with', 'two_with',
                                    'claw_attacks', 'bite_attack', 'weapon_attack']):
        return 'simple_specific'

    # Count-based simple
    if any(p in patterns for p in ['count_two', 'count_three', 'count_four']):
        return 'simple_count'

    return 'unclassified'

multiattack_df['complexity_category'] = multiattack_df.apply(categorize_multiattack, axis=1)

# Summary by category
category_counts = multiattack_df['complexity_category'].value_counts()
print("\nMultiattack Complexity Categories:")
print(category_counts.to_string())


Multiattack Complexity Categories:
complexity_category
simple_specific    83
unclassified       20
or_alternative     16
simple_count       11
conditional         5
variable_count      1
constrained         1


In [112]:
# Show examples from each category
print("=" * 70)
print("MULTIATTACK EXAMPLES BY CATEGORY")
print("=" * 70)

for category in ['variable_count', 'or_alternative', 'form_dependent',
                 'conditional', 'constrained', 'generic', 'simple_specific']:
    cat_df = multiattack_df[multiattack_df['complexity_category'] == category]
    if len(cat_df) > 0:
        print(f"\n### {category.upper()} ({len(cat_df)} creatures) ###")
        for _, row in cat_df.head(3).iterrows():
            print(f"\n[{row['monster']}]:")
            print(f"  {row['description']}")

MULTIATTACK EXAMPLES BY CATEGORY

### VARIABLE_COUNT (1 creatures) ###

[Hydra]:
  The hydra makes as many bite attacks as it has heads.

### OR_ALTERNATIVE (16 creatures) ###

[Bandit Captain]:
  The captain makes three melee attacks: two with its scimitar and one with its dagger. Or the captain makes two ranged attacks with its daggers.

[Centaur]:
  The centaur makes two attacks: one with its pike and one with its hooves or two with its longbow.

[Chimera]:
  The chimera makes three attacks: one with its bite, one with its horns, and one with its claws. When its fire breath is available, it can use the breath in place of its bite or horns.

### CONDITIONAL (5 creatures) ###

[Chuul]:
  The chuul makes two pincer attacks. If the chuul is grappling a creature, the chuul can also use its tentacles once.

[Grick]:
  The grick makes one attack with its tentacles. If that attack hits, the grick can make one beak attack against the same target.

[Half-Red Dragon Veteran]:
  The veteran mak

In [113]:
# Save multiattack analysis
multiattack_df['ma_patterns_str'] = multiattack_df['ma_patterns'].apply(lambda x: '|'.join(x) if x else '')

multiattack_df.to_csv(LOCAL_DATA_DIR + '/multiattack_analysis.csv', index=False)
print(f"\nSaved multiattack analysis: {len(multiattack_df)} rows")

# Save pattern frequency
ma_pattern_freq_df.to_csv(LOCAL_DATA_DIR + '/multiattack_pattern_frequencies.csv', index=False)
print(f"Saved multiattack pattern frequencies: {len(ma_pattern_freq_df)} patterns")

# Summary
print("\n" + "=" * 70)
print("MULTIATTACK ANALYSIS SUMMARY")
print("=" * 70)
print(f"\nTotal multiattack creatures: {len(multiattack_df)}")
print(f"With potentially problematic patterns: {multiattack_df['has_parsing_issue'].sum()} ({100*multiattack_df['has_parsing_issue'].mean():.1f}%)")
print(f"\nComplexity distribution:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count} ({100*count/len(multiattack_df):.1f}%)")


Saved multiattack analysis: 137 rows
Saved multiattack pattern frequencies: 15 patterns

MULTIATTACK ANALYSIS SUMMARY

Total multiattack creatures: 137
With potentially problematic patterns: 23 (16.8%)

Complexity distribution:
  simple_specific: 83 (60.6%)
  unclassified: 20 (14.6%)
  or_alternative: 16 (11.7%)
  simple_count: 11 (8.0%)
  conditional: 5 (3.6%)
  variable_count: 1 (0.7%)
  constrained: 1 (0.7%)
